# Compile-Time Analytics

This notebook helps you:

1. Run `ci/build_compile_time_bench.sh` from the repo root.
2. Load and inspect an all-header processing CSV.
3. Explore high-impact headers by TU coverage and average processing time.
4. Build combined ranking scores to identify optimization targets.

Expected CSV columns include:
- `header_path`
- `include_tu_count`
- `avg_process_time_s`
- `total_process_time_s`


In [ ]:
%pip install pandas matplotlib plotly nbformat

In [ ]:
import os
import shlex
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(
        "Could not locate repo root (.git). Start notebook from inside the CCCL repo."
    )

print(f"Repo root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")

In [ ]:
# --- Run build_compile_time_bench.sh (file-processing mode) ---
import subprocess
from pathlib import Path

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
        REPO_ROOT = REPO_ROOT.parent

output_csv = Path(os.environ.get("COMPILE_TIME_CSV", "/tmp/compile_time.csv"))
cmd = [
    "bash",
    str(REPO_ROOT / "ci" / "build_compile_time_bench.sh"),
    *shlex.split(os.environ.get("COMPILE_TIME_BUILD_ARGS", "")),
    "--",
    "-f",
    "file-processing",
    "-e",
    "-n",
    os.environ.get("COMPILE_TIME_TOP_N", "5000"),
    "--sort",
    "total",
    "--output-csv",
    str(output_csv),
]

print("Command:")
print("  " + " ".join(shlex.quote(x) for x in cmd))

subprocess.run(cmd, cwd=REPO_ROOT, check=True)
print(f"\nWrote: {output_csv}")

In [ ]:
# --- Load CSV ---
csv_path = Path(os.environ.get("COMPILE_TIME_CSV", "/tmp/compile_time.csv"))
if not csv_path.exists():
    raise FileNotFoundError(f"Missing CSV: {csv_path}. Run the script first.")

df = pd.read_csv(csv_path)

event_summary_cols = {
    "event_name",
    "event_key",
    "root_tu_count",
    "selected_avg_per_root_tu_s",
    "selected_total_s",
}
required_cols = [
    "header_path",
    "include_tu_count",
    "avg_process_time_s",
    "total_process_time_s",
]

if event_summary_cols.issubset(df.columns):
    df["header_path"] = df["event_key"]
    df["include_tu_count"] = df["root_tu_count"]
    if (
        "avg_inclusive_per_root_tu_s" in df.columns
        and "total_inclusive_s" in df.columns
    ):
        df["avg_process_time_s"] = df["avg_inclusive_per_root_tu_s"]
        df["total_process_time_s"] = df["total_inclusive_s"]
    else:
        df["avg_process_time_s"] = df["selected_avg_per_root_tu_s"]
        df["total_process_time_s"] = df["selected_total_s"]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(
        f"CSV is not all-header processing output. Missing columns: {missing}. "
        "Run the script cell above to regenerate /tmp/compile_time.csv."
    )

for col in ["include_tu_count", "avg_process_time_s", "total_process_time_s"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(5)

In [ ]:
# --- Quick top-N views ---
TOP_N = 5

print("Top by avg_process_time_s")
display(
    df.nlargest(TOP_N, "avg_process_time_s")[
        [
            "header_path",
            "avg_process_time_s",
            "include_tu_count",
            "total_process_time_s",
        ]
    ]
)

print("\nTop by include_tu_count")
display(
    df.nlargest(TOP_N, "include_tu_count")[
        [
            "header_path",
            "include_tu_count",
            "avg_process_time_s",
            "total_process_time_s",
        ]
    ]
)

print("\nTop by impact score (include_tu_count * avg_process_time_s)")
df_score = df.copy()
df_score["impact_score"] = df_score["include_tu_count"] * df_score["avg_process_time_s"]
display(
    df_score.nlargest(TOP_N, "impact_score")[
        [
            "header_path",
            "impact_score",
            "include_tu_count",
            "avg_process_time_s",
            "total_process_time_s",
        ]
    ]
)

### How to read this plot

- Each point is one header seen in all-mode profiling.
- **X axis (`include_tu_count`)**: how many generated public-header TUs include this header at least once.
- **Y axis (`avg_process_time_s`)**: average time spent processing that header per including TU.
- Headers near the **upper-right** are usually the best optimization candidates because they are both widespread and expensive per include.

In [ ]:
# --- Interactive scatter (hover shows header name) ---

import plotly.express as px

plot_df = df.copy()

total_headers = len(plot_df)
total_public_headers = int(plot_df["include_tu_count"].max())

fig = px.scatter(
    plot_df,
    x="include_tu_count",
    y="avg_process_time_s",
    hover_name="header_path",
    hover_data={
        "include_tu_count": True,
        "avg_process_time_s": ":.6f",
        "total_process_time_s": ":.3f",
        "header_path": False,
    },
    opacity=0.6,
    title=(
        f"Header include count vs avg processing time ({total_headers:,} total headers; "
        f"coverage measured across {total_public_headers} public headers)"
    ),
)

fig.update_layout(
    xaxis_title=(
        "TU coverage: number of public headers that include this header "
        f"(out of {total_public_headers})"
    ),
    yaxis_title="Average processing time per including TU (seconds)",
    height=900,
)
fig.show()

In [ ]:
# --- Optional: run ctadvisor and parse expensive headers ---
run_ctadvisor = os.environ.get("COMPILE_TIME_RUN_CTADVISOR") == "1"
CTADVISOR_ENTRIES = 10
CTADVISOR_THREADS = os.cpu_count() or 8

trace_root = (
    REPO_ROOT
    / "build"
    / os.environ.get("CCCL_BUILD_INFIX", "cuda13.1-gcc14")
    / os.environ.get("CCCL_COMPILE_TIME_PRESET", "all-dev")
    / "compile_time"
    / "raw_traces"
)
ctadvisor_cmd = [
    "ctadvisor",
    "--trace-file-path",
    str(trace_root),
    "--header-advisor-entries",
    str(CTADVISOR_ENTRIES),
    "--thread-number",
    str(CTADVISOR_THREADS),
]

if run_ctadvisor:
    print("Command:")
    print("  " + " ".join(shlex.quote(x) for x in ctadvisor_cmd))

    result = subprocess.run(
        ctadvisor_cmd, cwd=REPO_ROOT, check=True, capture_output=True, text=True
    )
    print(result.stdout)
else:
    print("Skipping ctadvisor. Set COMPILE_TIME_RUN_CTADVISOR=1 to run it.")